In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os

In [23]:
from pathlib import Path

csv_folder = Path.cwd()
dataframes = {
    csv_file.stem: pd.read_csv(csv_file)
    for csv_file in csv_folder.glob("*.csv")
}

if not dataframes:
    raise FileNotFoundError(f"No CSV files found in {csv_folder}")

for name, dataframe in dataframes.items():
    globals()[name] = dataframe
    print(f"{name}: {dataframe.shape[0]} rows, {dataframe.shape[1]} columns")


dataframes['accounts']
dataframes["data_dictionary"]
dataframes["sales_pipeline"]

accounts: 85 rows, 7 columns
data_dictionary: 21 rows, 3 columns
products: 7 rows, 3 columns
sales_pipeline: 8800 rows, 8 columns
sales_teams: 35 rows, 3 columns


,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0
...,...,...,...,...,...,...,...,...
8795,9MIWFW5J,Versie Hillebrand,MG Advanced,NaN,Prospecting,NaN,NaN,NaN
8796,6SLKZ8FI,Versie Hillebrand,MG Advanced,NaN,Prospecting,NaN,NaN,NaN
8797,LIB4KUZJ,Versie Hillebrand,MG Advanced,NaN,Prospecting,NaN,NaN,NaN
8798,18IUIUK0,Versie Hillebrand,MG Advanced,NaN,Prospecting,NaN,NaN,NaN


In [24]:
#accounts

df=dataframes['accounts']

df =df.drop(columns=["subsidiary_of"])

print(df)




             account         sector  year_established  revenue  employees  \
0   Acme Corporation      technolgy              1996  1100.04       2822   
1         Betasoloin        medical              1999   251.41        495   
2           Betatech        medical              1986   647.18       1185   
3         Bioholding        medical              2012   587.34       1356   
4            Bioplex        medical              1991   326.82       1016   
..               ...            ...               ...      ...        ...   
80        Zathunicon         retail              2010    71.12        144   
81    Zencorporation      technolgy              2011    40.79        142   
82            Zoomit  entertainment              1992   324.19        978   
83           Zotware       software              1979  4478.47      13809   
84          Zumgoity        medical              1984   441.08       1210   

   office_location  
0    United States  
1    United States  
2           

In [31]:
# Merge tables (replicating your star schema model)

df = dataframes["sales_pipeline"].merge(
    dataframes["accounts"],
    on="account",
    how="left"
)

df = df.merge(
    dataframes["sales_teams"],
    on="sales_agent",
    how="left"
)

print(df.head(10))


  opportunity_id      sales_agent         product   account deal_stage  \
0       1C1I7A6R      Moses Frase  GTX Plus Basic   Cancity        Won   
1       Z063OYW0  Darcel Schlecht          GTXPro     Isdom        Won   
2       EC4QE1BX  Darcel Schlecht      MG Special   Cancity        Won   
3       MV1LWRNH      Moses Frase       GTX Basic   Codehow        Won   
4       PE84CX4O        Zane Levy       GTX Basic    Hatfan        Won   
5       ZNBS69V1    Anna Snelling      MG Special  Ron-tech        Won   
6       9ME3374G   Vicki Laflamme      MG Special   J-Texon        Won   
7       7GN8Q4LL   Markita Hansen       GTX Basic    Cheers        Won   
8       OLK9LKZB  Niesha Huffines  GTX Plus Basic  Zumgoity        Won   
9       HAXMC4IX   James Ascencio     MG Advanced       NaN   Engaging   

  engage_date  close_date  close_value         sector  year_established  \
0  2016-10-20  2017-03-01       1054.0         retail            2001.0   
1  2016-10-25  2017-03-11       451

In [32]:
# Convert date columns to datetime
df['engage_date'] = pd.to_datetime(df['engage_date'])
df['close_date'] = pd.to_datetime(df['close_date'])


print(df.dtypes)

print(df.describe(include='all'))

print(df.isnull().sum())

opportunity_id                 str
sales_agent                    str
product                        str
account                        str
deal_stage                     str
engage_date         datetime64[us]
close_date          datetime64[us]
close_value                float64
sector                         str
year_established           float64
revenue                    float64
employees                  float64
office_location                str
subsidiary_of                  str
manager                        str
regional_office                str
dtype: object
       opportunity_id      sales_agent    product   account deal_stage  \
count            8800             8800       8800      7375       8800   
unique           8800               30          7        85          4   
top          1C1I7A6R  Darcel Schlecht  GTX Basic  Hottechi        Won   
freq                1              747       1866       200       4238   
mean              NaN              NaN        NaN       

Step 2: Feature Engineering (Replacing DAX Measures)

In [33]:
df["won_flag"] = (df["deal_stage"] == "Won").astype(int)


df['total_revenue'] = df.apply(lambda row: row['close_value'] if row['won_flag'] == 1 else 0, axis=1)

df['sales_cycle_days'] = (df['close_date'] - df['engage_date']).dt.days


# 4. Agent Performance Summary Table
agent_summary = df.groupby('sales_agent').agg(
    total_revenue=('total_revenue', 'sum'),
    total_deals=('opportunity_id', 'count'),
    won_deals=('won_flag', 'sum')
).reset_index()

agent_summary['win_rate'] = agent_summary['won_deals'] / agent_summary['total_deals']
agent_summary = agent_summary.sort_values(by='total_revenue', ascending=False)


agent_summary



,sales_agent,total_revenue,total_deals,won_deals,win_rate
6,Darcel Schlecht,1153214.0,747,349,0.467202
26,Vicki Laflamme,478396.0,451,221,0.490022
15,Kary Hendrixson,454298.0,438,209,0.477169
2,Cassey Cress,450489.0,346,163,0.471098
7,Donn Cantrell,445860.0,275,158,0.574545
22,Reed Clapper,438336.0,237,155,0.654008
29,Zane Levy,430068.0,349,161,0.461318
4,Corliss Cosme,421036.0,310,150,0.483871
12,James Ascencio,413533.0,267,135,0.505618
5,Daniell Hammack,364229.0,259,114,0.440154


Step 3: Build the Visualizations (Using Plot)

In [28]:
funnel_data = df.groupby('deal_stage')['opportunity_id'].count().reset_index()
# Define custom order if needed: Prospecting -> Engaging -> Lost/Won
import plotly.express as px

fig_funnel = px.funnel(
    funnel_data,
    x="opportunity_id",
    y="deal_stage",
    title="Pipeline Drop-off by Stage",
    category_orders={
        "deal_stage": ["Prospecting", "Engaging", "Lost", "Won"]
    }
)

fig_funnel.show()

In [29]:
fig_agent = px.bar(
    agent_summary.sort_values('total_revenue', ascending=True), 
    x='total_revenue', 
    y='sales_agent', 
    orientation='h',
    title='Total Revenue by Sales Agent'
)
fig_agent.show()

In [35]:
sector_revenue = df.groupby('sector')['total_revenue'].sum().reset_index()
fig_treemap = px.treemap(
    sector_revenue, 
    path=['sector'], 
    values='total_revenue', 
    title='Total Revenue by Sector'
)
fig_treemap.show()

In [34]:
customer_summary = df.groupby(['account', 'revenue']).agg(
    total_sales=('total_revenue', 'sum')
).reset_index()

fig_scatter = px.scatter(
    customer_summary, 
    x='revenue', 
    y='total_sales', 
    hover_name='account',
    title='Customer Revenue vs. Total Sales'
)
fig_scatter.show()